# Environment Setup

Import required libraries and initialize CI/CD configuration.

In [2]:
# Import libraries and configure project settings

import warnings
warnings.filterwarnings("ignore")

import json
import boto3
import sagemaker
import pandas as pd

PROJECT_PREFIX = "loanwise"

PIPELINE_NAME = "LoanApprovalPipeline"

MINIMUM_F1_SCORE = 0.65

sess = sagemaker.Session()

bucket = sess.default_bucket()

print(f"SageMaker Version : {sagemaker.__version__}")
print(f"Bucket            : {bucket}")

SageMaker Version : 2.224.4
Bucket            : sagemaker-us-east-1-612256167011


# Load Pipeline Inputs

Load datasets and evaluation artifacts required for pipeline validation.

In [3]:
# Load project artifacts

train_df = pd.read_csv(
    "processed_train.csv"
)

with open(
    "evaluation_metrics.json",
    "r"
) as f:

    metrics_df = pd.read_json(f)

print(f"Training Shape : {train_df.shape}")

display(metrics_df)

Training Shape : (16000, 18)


,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,XGBoost,0.67125,0.672024,0.6690,0.670509,0.67125
1,Random Forest,0.67175,0.678627,0.6525,0.665307,0.67175
2,Logistic Regression,0.66450,0.673158,0.6395,0.655897,0.66450


# Data Validation Checkpoint

Validate dataset availability and required columns.

In [4]:
# Data validation

required_columns = [
    "SK_ID_CURR",
    "TARGET"
]

missing_columns = [
    col
    for col in required_columns
    if col not in train_df.columns
]

data_validation_status = (
    len(missing_columns) == 0
)

print(
    f"Data Validation Status : "
    f"{data_validation_status}"
)

Data Validation Status : True


# Feature Validation Checkpoint

Validate engineered features and data quality.

In [5]:
# Feature validation

feature_count = len(train_df.columns)

feature_validation_status = (
    feature_count >= 10
)

print(
    f"Feature Count : {feature_count}"
)

print(
    f"Feature Validation Status : "
    f"{feature_validation_status}"
)

Feature Count : 18
Feature Validation Status : True


# Model Validation Checkpoint

Validate model artifact availability.

In [6]:
# Model validation

import os

model_validation_status = os.path.exists(
    "best_model.joblib"
)

print(
    f"Model Validation Status : "
    f"{model_validation_status}"
)

Model Validation Status : True


# Evaluation Validation Checkpoint

Validate model performance against deployment thresholds.

In [7]:
# Evaluation validation

best_f1_score = metrics_df["F1"].max()

evaluation_validation_status = (
    best_f1_score >= MINIMUM_F1_SCORE
)

print(
    f"Best F1 Score : {best_f1_score:.4f}"
)

print(
    f"Evaluation Validation Status : "
    f"{evaluation_validation_status}"
)

Best F1 Score : 0.6705
Evaluation Validation Status : True


# Deployment Approval Checkpoint

Determine whether the model is approved for deployment.

In [8]:
# Deployment approval

deployment_approved = all([
    data_validation_status,
    feature_validation_status,
    model_validation_status,
    evaluation_validation_status
])

approval_status = (
    "Approved"
    if deployment_approved
    else "Rejected"
)

print(
    f"Deployment Status : "
    f"{approval_status}"
)

Deployment Status : Approved


# Create Pipeline DAG

Generate a CI/CD pipeline representation.

In [9]:
# Create pipeline DAG metadata

pipeline_dag = {
    "Pipeline": PIPELINE_NAME,
    "Steps": [
        "Data Validation",
        "Feature Validation",
        "Model Validation",
        "Evaluation Validation",
        "Deployment Approval"
    ],
    "Status": approval_status
}

with open(
    "pipeline_dag.json",
    "w"
) as f:

    json.dump(
        pipeline_dag,
        f,
        indent=4
    )

display(pipeline_dag)

{'Pipeline': 'LoanApprovalPipeline',
 'Steps': ['Data Validation',
  'Feature Validation',
  'Model Validation',
  'Evaluation Validation',
  'Deployment Approval'],
 'Status': 'Approved'}

# Upload Pipeline Artifacts

Upload CI/CD outputs to Amazon S3.

In [10]:
# Upload CI/CD artifacts

pipeline_s3_uri = sess.upload_data(
    path="pipeline_dag.json",
    bucket=bucket,
    key_prefix=f"{PROJECT_PREFIX}/cicd"
)

print(pipeline_s3_uri)

s3://sagemaker-us-east-1-612256167011/loanwise/cicd/pipeline_dag.json


# Verify CI/CD Outputs

Review pipeline artifacts and deployment decision.

In [11]:
# Verify CI/CD outputs

print("Pipeline Name")
print("-------------")
print(PIPELINE_NAME)

print("\nDeployment Status")
print("-----------------")
print(approval_status)

print("\nPipeline Artifact")
print("-----------------")
print(pipeline_s3_uri)

display(pipeline_dag)

print("\nNotebook 6 completed successfully.")

Pipeline Name
-------------
LoanApprovalPipeline

Deployment Status
-----------------
Approved

Pipeline Artifact
-----------------
s3://sagemaker-us-east-1-612256167011/loanwise/cicd/pipeline_dag.json


{'Pipeline': 'LoanApprovalPipeline',
 'Steps': ['Data Validation',
  'Feature Validation',
  'Model Validation',
  'Evaluation Validation',
  'Deployment Approval'],
 'Status': 'Approved'}


Notebook 6 completed successfully.
